<a href="https://colab.research.google.com/github/AvivBE-IS/Demo-Project/blob/main/Video_to_Text.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Install required libraries for transcription (faster-whisper) and audio handling
!pip install faster-whisper
!pip install ffmpeg-python

In [2]:
from google.colab import drive

# Mount your Google Drive account. You will be prompted to grant access.
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
from faster_whisper import WhisperModel
import time

# --- Path Definitions ---
# Ensure the paths match the exact location in your Drive.
# Based on the images, the folder is in My Drive.
base_dir = '/content/drive/MyDrive'

source_dir = os.path.join(base_dir, 'Records')
output_dir = os.path.join(base_dir, 'Records', 'transcripts')

# Create the destination folder if it doesn't exist
if not os.path.exists(output_dir):
    os.makedirs(output_dir)
    print(f"Created output directory: {output_dir}")
else:
    print(f"Output directory already exists: {output_dir}")

In [ ]:
# Initialize the Whisper model.
# Using the 'large-v3' model for maximum accuracy (recommended if GPU is available).
# If out of memory, you can change this to 'medium' or 'small'.
model_size = "large-v3"

print(f"Loading Whisper model ({model_size})...")
# It is recommended to run this notebook with a GPU (Runtime -> Change runtime type -> T4 GPU)
model = WhisperModel(model_size, device="cuda", compute_type="float16")
print("Model loaded successfully.")

In [3]:
def transcribe_video(video_path, txt_path):
    print(f"  Transcribing: {os.path.basename(video_path)}...")
    start_time = time.time()

    # Transcribe the audio. Note the language parameter (he = Hebrew).
    segments, info = model.transcribe(video_path, beam_size=5, language="he")

    print(f"  Detected language '{info.language}' with probability {info.language_probability}")

    # Write to the file while iterating through the segments
    with open(txt_path, "w", encoding="utf-8") as f:
        for segment in segments:
            # Optional: add timestamps -> f.write(f"[{segment.start:.2f}s -> {segment.end:.2f}s] {segment.text}\n")
            f.write(segment.text + "\n")

    end_time = time.time()
    print(f"  Finished in {end_time - start_time:.2f} seconds. Saved to: {os.path.basename(txt_path)}")

# --- Run transcription on all files ---
print(f"Starting processing in directory: {source_dir}")

# Iterate through all files in the source directory
for filename in os.listdir(source_dir):
    file_path = os.path.join(source_dir, filename)

    # Ensure it is a file and not a directory (ignores the 'transcripts' directory)
    if os.path.isfile(file_path):
        # Check if the file is a video or audio file
        if filename.lower().endswith(('.mp4', '.mkv', '.avi', '.mov', '.mp3', '.m4a')):

            # Create the output filename inside the destination directory
            base_name = os.path.splitext(filename)[0]
            txt_filename = base_name + ".txt"
            txt_path = os.path.join(output_dir, txt_filename)

            # Check if the file has already been transcribed
            if os.path.exists(txt_path):
                 print(f"Skipping {filename} - transcript already exists.")
            else:
                 print(f"\nProcessing file: {filename}")
                 transcribe_video(file_path, txt_path)

print("\n--- All done! ---")

Found 7 MP4 file(s) in '/content/drive/MyDrive/Records'.
Loading Whisper model (large-v3) on GPU...



[1/7] Processing: Lecture_01_2025-10-29...
  -> SUCCESS: Saved to Drive: Lecture_01_2025-10-29.txt (3354 lines, Last: [02:19:40], 173.2 KB)

[2/7] Processing: Lecture_02_2025-11-05...
  -> SUCCESS: Saved to Drive: Lecture_02_2025-11-05.txt (3167 lines, Last: [02:23:11], 178.5 KB)

[3/7] Processing: Lecture_03_2026-10-21_Part1...
  -> SUCCESS: Saved to Drive: Lecture_03_2026-10-21_Part1.txt (1077 lines, Last: [00:44:31], 56.3 KB)

[4/7] Processing: Lecture_04_2026-10-21_Part2...
  [Warning] Audio track is silent for Lecture_04_2026-10-21_Part2. Skipping transcription.

[5/7] Processing: Lecture_05_2026-10-28...
  -> SUCCESS: Saved to Drive: Lecture_05_2026-10-28.txt (3151 lines, Last: [02:09:03], 160.1 KB)

[6/7] Processing: Lecture_06_2026-11-04...
  -> SUCCESS: Saved to Drive: Lecture_06_2026-11-04.txt (3321 lines, Last: [02:19:08], 167.1 KB)

[7/7] Processing: Lecture_07_2026-11-11...
  -> SUCCESS: Saved to Drive: Lecture_07_2026-11-11.txt (2454 lines, Last: [01:47:11], 129.2 KB)

D